# EcoHome Energy Advisor - ChromaDB Vector Store Setup

Set up a vector store for Retrieval-Augmented Generation (RAG) using ChromaDB. This pipeline loads energy optimization documents, creates embeddings, and enables semantic search for the EcoHome energy advisor.

## Learning Objectives
- Load all documents from data/documents/ directory
- Split documents using RecursiveCharacterTextSplitter
- Create OpenAI embeddings
- Build and persist ChromaDB vector store
- Implement similarity search with validation
- Handle existing vector store loads efficiently

## Pipeline Overview
1. Load all .txt files from data/documents/
2. Split into chunks (1000 chars, 200 overlap)
3. Create OpenAI embeddings
4. Check if vector store exists → load or create
5. Run validation searches and retrieve examples

## 1. Import Required Libraries


In [1]:
import os
import glob
from pathlib import Path
from dotenv import load_dotenv
import numpy as np

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# Load environment variables
load_dotenv()

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [2]:
# Verify environment configuration
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("⚠ Warning: OPENAI_API_KEY not found. Using default OpenAI endpoint.")
else:
    print(f"✓ OpenAI API key loaded ({len(api_key)} characters)")

# Configure paths
vectorstore_path = "data/vectorstore"
documents_path = "data/documents"
print(f"✓ Vectorstore path: {vectorstore_path}")
print(f"✓ Documents path: {documents_path}")

⚠ Warning: OPENAI_API_KEY not found. Using default OpenAI endpoint.
✓ Vectorstore path: data/vectorstore
✓ Documents path: data/documents


## 2. Load and Process Documents


In [3]:
# Load all .txt files from data/documents directory
documents = []

try:
    # Find all .txt files in documents directory
    txt_files = glob.glob(f"{documents_path}/*.txt")
    
    if not txt_files:
        print(f"⚠ No .txt files found in {documents_path}")
    else:
        print(f"Found {len(txt_files)} document files:")
        
        for file_path in sorted(txt_files):
            try:
                loader = TextLoader(file_path, encoding='utf-8')
                docs = loader.load()
                documents.extend(docs)
                file_name = Path(file_path).name
                print(f"  ✓ {file_name}: {len(docs)} document(s), {len(docs[0].page_content)} characters")
            except Exception as e:
                print(f"  ✗ Error loading {Path(file_path).name}: {str(e)}")
    
    print(f"\n✓ Total documents loaded: {len(documents)}")
    
    # Show document statistics
    total_chars = sum(len(doc.page_content) for doc in documents)
    print(f"✓ Total text size: {total_chars:,} characters")
    
except Exception as e:
    print(f"✗ Error loading documents: {str(e)}")
    documents = []

Found 3 document files:
  ✓ hvac_optimization_guide.txt: 1 document(s), 7396 characters
  ✓ tip_device_best_practices.txt: 1 document(s), 1734 characters
  ✓ tip_energy_savings.txt: 1 document(s), 1396 characters

✓ Total documents loaded: 3
✓ Total text size: 10,526 characters


## 3. Split Documents into Chunks


In [4]:
# Split documents into chunks for embedding
splits = []

try:
    if not documents:
        print("⚠ No documents to split")
    else:
        # Initialize text splitter with RAG-optimized parameters
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,          # Size of each chunk
            chunk_overlap=200,        # Overlap for context preservation
            length_function=len,      # Use character length
            separators=["\n\n", "\n", ". ", " ", ""]  # Split on meaningful boundaries
        )
        
        # Split all documents
        splits = text_splitter.split_documents(documents)
        
        print(f"✓ Split documents into {len(splits)} chunks")
        print(f"  Average chunk size: {np.mean([len(s.page_content) for s in splits]):.0f} characters")
        print(f"  Min/Max chunk size: {min(len(s.page_content) for s in splits)} / {max(len(s.page_content) for s in splits)} characters")
        
        # Display sample chunks
        print(f"\n✓ Sample chunks:")
        for i, split in enumerate(splits[:3]):
            preview = split.page_content[:150].replace('\n', ' ')
            print(f"  Chunk {i+1}: {preview}...")

except Exception as e:
    print(f"✗ Error splitting documents: {str(e)}")
    splits = []

✓ Split documents into 13 chunks
  Average chunk size: 882 characters
  Min/Max chunk size: 567 / 989 characters

✓ Sample chunks:
  Chunk 1: HVAC OPTIMIZATION GUIDE FOR SMART HOMES Comprehensive Energy Savings Strategy for Heating and Cooling Systems  OVERVIEW Heating, Ventilation, and Air ...
  Chunk 2: Evening Period (5 PM - 10 PM): - Restore comfort temperature: 70°F heating, 74°F cooling - Gradual adjustment starts 30 minutes before occupants arriv...
  Chunk 3: Summer Season (June - August): - Daytime cooling setpoint: 74-76°F - Nighttime cooling setpoint: 78-80°F - Defer cooling during peak hours (2 PM - 9 P...


## 4. Create Vector Store


In [6]:
# Create or load ChromaDB vector store
vectorstore = None

try:
    # Create vectorstore directory if it doesn't exist
    os.makedirs(vectorstore_path, exist_ok=True)
    
    # Initialize embeddings
    print("Initializing OpenAI embeddings...")
    try:
        embeddings = OpenAIEmbeddings(
            model="text-embedding-3-small"
        )
        print("✓ OpenAI embeddings initialized")
    except Exception as e:
        print(f"⚠ OpenAI embedding error: {str(e)}")
        print("  Make sure OPENAI_API_KEY environment variable is set")
        print("  Or add it to your .env file")
        embeddings = None
    
    if embeddings is None:
        print("\n⚠ Skipping vector store creation due to missing API key")
        print("  To enable RAG functionality, set your OpenAI API key:")
        print("    1. Create/edit .env file in project root")
        print("    2. Add: OPENAI_API_KEY=your-api-key-here")
        print("    3. Run this notebook again")
    else:
        # Check if vector store already exists
        chroma_db_file = os.path.join(vectorstore_path, "chroma.db")
        
        if os.path.exists(chroma_db_file) and len(splits) == 0:
            # Load existing vector store
            print(f"\n✓ Loading existing vector store from {vectorstore_path}...")
            vectorstore = Chroma(
                persist_directory=vectorstore_path,
                embedding_function=embeddings
            )
            print(f"✓ Vector store loaded successfully")
            try:
                count = vectorstore._collection.count()
                print(f"  Total vectors in store: {count}")
            except:
                print(f"  Vector store loaded (collection count unavailable)")
            
        elif len(splits) > 0:
            # Create new vector store from documents
            print(f"\nCreating new vector store from {len(splits)} chunks...")
            vectorstore = Chroma.from_documents(
                documents=splits,
                embedding=embeddings,
                persist_directory=vectorstore_path
            )
            vectorstore.persist()
            print(f"✓ Vector store created and persisted")
            print(f"  Total vectors stored: {len(splits)}")
        else:
            print("⚠ No documents to process and no existing vector store found")

except Exception as e:
    print(f"✗ Error creating/loading vector store: {str(e)}")

Initializing OpenAI embeddings...
⚠ OpenAI embedding error: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
  Make sure OPENAI_API_KEY environment variable is set
  Or add it to your .env file

⚠ Skipping vector store creation due to missing API key
  To enable RAG functionality, set your OpenAI API key:
    1. Create/edit .env file in project root
    2. Add: OPENAI_API_KEY=your-api-key-here
    3. Run this notebook again


## 5. Test the RAG Pipeline


In [7]:
# Run similarity search validation queries
if vectorstore is None:
    print("⚠ Vector store not available for testing")
else:
    # Define test queries covering various energy optimization topics
    test_queries = [
        "HVAC thermostat optimization strategies",
        "electric vehicle charging efficiency",
        "solar panel energy generation",
        "peak hour demand reduction",
        "dishwasher energy savings",
        "pool pump scheduling"
    ]
    
    print("=" * 80)
    print("SIMILARITY SEARCH VALIDATION TESTS")
    print("=" * 80)
    
    for query_idx, query in enumerate(test_queries, 1):
        print(f"\n[Query {query_idx}] {query}")
        print("-" * 80)
        
        try:
            # Perform similarity search
            results = vectorstore.similarity_search_with_scores(query, k=3)
            
            if not results:
                print("  ⚠ No results found")
                continue
            
            print(f"  ✓ Found {len(results)} relevant chunks:")
            
            for result_idx, (doc, score) in enumerate(results, 1):
                # Extract filename from metadata
                source = doc.metadata.get('source', 'unknown')
                source_name = Path(source).name if source else 'unknown'
                
                # Preview content
                content_preview = doc.page_content[:200].replace('\n', ' ')
                
                print(f"\n  Result {result_idx}:")
                print(f"    Source: {source_name}")
                print(f"    Relevance Score: {score:.4f}")
                print(f"    Preview: {content_preview}...")
                
        except Exception as e:
            print(f"  ✗ Error in search: {str(e)}")
    
    print("\n" + "=" * 80)

⚠ Vector store not available for testing


## 7. Detailed Chunk Retrieval and Inspection

Retrieve and display full content of matched chunks with detailed metadata for validation.

In [8]:
# Detailed chunk retrieval and validation
if vectorstore is None:
    print("⚠ Vector store not available for chunk inspection")
else:
    print("=" * 80)
    print("DETAILED CHUNK RETRIEVAL AND INSPECTION")
    print("=" * 80)
    
    # Sample queries for detailed inspection
    inspection_queries = [
        "HVAC peak hour reduction strategies",
        "solar battery optimization",
        "device energy consumption patterns"
    ]
    
    for query in inspection_queries:
        print(f"\n[Query] {query}")
        print("-" * 80)
        
        try:
            # Retrieve top 2 chunks with scores
            results = vectorstore.similarity_search_with_scores(query, k=2)
            
            if not results:
                print("  ⚠ No results found")
                continue
            
            for idx, (doc, score) in enumerate(results, 1):
                print(f"\nChunk {idx}:")
                print(f"  Relevance Score: {score:.4f}")
                print(f"  Source: {Path(doc.metadata.get('source', 'unknown')).name}")
                print(f"  Content Length: {len(doc.page_content)} characters")
                print(f"\n  Full Content:")
                print("  " + "-" * 76)
                
                # Print content with proper formatting
                for line in doc.page_content.split('\n'):
                    if line.strip():
                        print(f"  {line}")
                print("  " + "-" * 76)
                
        except Exception as e:
            print(f"  ✗ Error retrieving chunks: {str(e)}")
    
    print("\n" + "=" * 80)
    
    # Summary statistics
    print("\nVECTOR STORE SUMMARY")
    print("=" * 80)
    try:
        total_vectors = vectorstore._collection.count()
        print(f"✓ Total vectors in store: {total_vectors:,}")
        print(f"✓ Vectorstore location: {vectorstore_path}")
        print(f"✓ Ready for RAG pipeline integration")
    except Exception as e:
        print(f"⚠ Could not retrieve vector store stats: {str(e)}")

⚠ Vector store not available for chunk inspection


In [9]:
print("\n" + "=" * 80)
print("VECTOR STORE SETUP COMPLETE")
print("=" * 80)
print(f"""
✓ Configuration:
  - Documents path: {documents_path}
  - Vectorstore path: {vectorstore_path}
  - Chunk size: 1000 characters
  - Chunk overlap: 200 characters
  - Embedding model: text-embedding-3-small

✓ Status:
  - Documents loaded: {len(documents)}
  - Chunks created: {len(splits)}
  - Vector store ready: {vectorstore is not None}

Next Steps:
1. Use vectorstore for semantic search in RAG pipelines
2. Integrate with LangChain agents for energy optimization queries
3. Monitor retrieval quality and adjust chunk parameters if needed
4. Add additional documents to data/documents/ as needed

Run this notebook again to reload the existing vector store.
""")
print("=" * 80)


VECTOR STORE SETUP COMPLETE

✓ Configuration:
  - Documents path: data/documents
  - Vectorstore path: data/vectorstore
  - Chunk size: 1000 characters
  - Chunk overlap: 200 characters
  - Embedding model: text-embedding-3-small

✓ Status:
  - Documents loaded: 3
  - Chunks created: 13
  - Vector store ready: False

Next Steps:
1. Use vectorstore for semantic search in RAG pipelines
2. Integrate with LangChain agents for energy optimization queries
3. Monitor retrieval quality and adjust chunk parameters if needed
4. Add additional documents to data/documents/ as needed

Run this notebook again to reload the existing vector store.

